# Modello di Classificazione dei Pezzi Difettosi in Produzione
## AutomaParts S.p.A. — Linea di assemblaggio bracci trasversali

---

# RELAZIONE TECNICA

## 1. Analisi Preliminare del Problema

### Contesto aziendale

AutomaParts S.p.A. è un fornitore tier-1 del settore automotive che produce componenti meccanici di precisione per sistemi sterzo e sospensioni. La qualità dei pezzi è un requisito critico: un pezzo difettoso che raggiunge l'assemblaggio finale può causare fermi macchina, costi di rilavorazione elevati, o richiami da parte degli OEM con impatti economici e reputazionali significativi.

### Il problema di classificazione

Per ogni pezzo prodotto vengono registrate misure dimensionali (diametro, lunghezza, planaritá), parametri di processo (temperatura, coppia, vibrazione, tempo ciclo), informazioni sulla rugosità superficiale e un punteggio da ispezione visiva. A fine ciclo viene assegnata l'etichetta binaria `defect_label` (0 = conforme, 1 = difettoso).

**Obiettivo**: addestrare un modello di Machine Learning supervisionato capace di predire automaticamente se un pezzo è difettoso, basandosi sulle misure registrate in linea.

### Sfide specifiche

- **Sbilanciamento delle classi**: i pezzi difettosi sono la minoranza. Richiede metriche adeguate (F1, ROC-AUC), non la sola accuracy.
- **Costo asimmetrico degli errori**: un **falso negativo** (difettoso classificato come conforme) è molto più costoso di un **falso positivo** (conforme bloccato per controllo extra). La soglia di classificazione deve riflettere questo trade-off.
- **Dati misti**: variabili numeriche continue, categoriche e timestamp. Necessario un pre-processing coerente.
- **Interpretabilità**: il modello deve essere comprensibile agli ingegneri di processo per guidare interventi correttivi.

---

## 2. Giustificazione delle Scelte Tecniche e Algoritmiche

### Scelta dei modelli

Sono stati selezionati tre algoritmi con caratteristiche complementari:

| Modello | Motivo della scelta | Vantaggio | Limite |
|---|---|---|---|
| **Decision Tree** | Baseline interpretabile | Regole if-then leggibili da tecnici | Overfitting su dati rumorosi |
| **Random Forest** | Ensemble robusto | Alta performance, gestisce outlier | Meno interpretabile del singolo albero |
| **Logistic Regression** | Modello lineare probabilistico | Output probabilistico calibrato | Assume separabilità lineare |

### Gestione dei dati mancanti

- Variabili **numeriche**: imputazione con la **mediana** (robusta agli outlier).
- Variabili **categoriche**: imputazione con la **moda**.

### Feature engineering

- `production_timestamp` decompost in `hour_of_day`, `day_of_week`, `month` (proxy per turno di lavoro).
- `material_batch`: estratta la settimana di produzione (`batch_week`) come variabile numerica.
- `line_id`, `station_id`, `operator_id`: usati come variabili numeriche ordinali.

### Scaling e valutazione

- **StandardScaler** sulle variabili numeriche (obbligatorio per Logistic Regression).
- Split **stratificato 80/20** + **cross-validation a 5 fold**.
- Metriche principali: **ROC-AUC**, **F1-score** (macro), **Recall** sulla classe difettosa.

---

## 3. Conclusioni Attese

- Il **Random Forest** dovrebbe ottenere le migliori performance (ROC-AUC più alto).
- Le feature più informative dovrebbero essere quelle dimensionali e di processo.
- La soglia ottimale andrà abbassata rispetto a 0.5 per privilegiare il recall sui difettosi.

---
---

# NOTEBOOK — Sviluppo del Modello

## Sezione 0 — Import delle Librerie

In [ ]:
# Installa eventuali librerie mancanti (utile su Google Colab)
# !pip install pandas numpy matplotlib seaborn scikit-learn -q

# ── Librerie standard ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# ── Scikit-learn: pre-processing ─────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# ── Scikit-learn: modelli ─────────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# ── Scikit-learn: metriche ────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    f1_score,
    ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

RANDOM_STATE = 42  # seed per riproducibilità

print('Librerie caricate correttamente.')

---
## Sezione 1 — Caricamento del Dataset

In [ ]:
# Percorso del file CSV relativo alla posizione del notebook.
# Su Google Colab: caricare il file manualmente oppure con !wget.
DATA_PATH = 'dataset/parts_production_data.csv'

df = pd.read_csv(DATA_PATH)
print(f'Dataset caricato: {df.shape[0]} righe x {df.shape[1]} colonne')
df.head()

In [ ]:
# Panoramica dei tipi di dati e valori non nulli
df.info()

In [ ]:
# Statistiche descrittive (scale, range, valori anomali)
df.describe().round(3)

---
## Sezione 2 — Analisi Esplorativa dei Dati (EDA)

L'EDA serve a capire la struttura del dataset, identificare anomalie e ottenere insight utili
per il business **prima** di costruire il modello.

### 2.1 Distribuzione della variabile target

In [ ]:
# Contiamo quanti pezzi conformi (0) e difettosi (1) ci sono.
# Con forte sbilanciamento la sola accuracy è fuorviante:
# un modello che predice sempre "0" avrebbe alta accuracy ma non rileva nessun difetto.
target_counts = df['defect_label'].value_counts()
target_pct    = df['defect_label'].value_counts(normalize=True) * 100

print('Distribuzione classe target:')
print(pd.DataFrame({'Conteggio': target_counts, 'Percentuale (%)': target_pct.round(2)}))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Conforme (0)', 'Difettoso (1)'], target_counts,
       color=['steelblue', 'tomato'], edgecolor='white')
ax.set_title('Distribuzione: conformi vs difettosi', fontsize=13)
ax.set_ylabel('Numero di pezzi')
for i, v in enumerate(target_counts):
    ax.text(i, v + 10, f'{v} ({target_pct.iloc[i]:.1f}%)', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

### 2.2 Tasso di difettosità per linea produttiva

In [ ]:
# La media di defect_label per linea equivale alla % di pezzi difettosi per linea.
# Utile per i responsabili di produzione: identifica le linee più critiche.
defect_by_line = (
    df.groupby('line_id')['defect_label']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'defect_rate', 'count': 'n_pezzi'})
      .sort_values('defect_rate', ascending=False)
)
defect_by_line['defect_pct'] = (defect_by_line['defect_rate'] * 100).round(2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(defect_by_line.index.astype(str), defect_by_line['defect_pct'],
       color='tomato', edgecolor='white')
ax.axhline(defect_by_line['defect_pct'].mean(), color='navy', linestyle='--',
           label=f'Media ({defect_by_line["defect_pct"].mean():.1f}%)')
ax.set_title('Tasso di difettosità per linea produttiva (%)', fontsize=13)
ax.set_xlabel('Linea ID')
ax.set_ylabel('% pezzi difettosi')
ax.legend()
plt.tight_layout()
plt.show()

print(defect_by_line.to_string())

### 2.3 Tasso di difettosità per settimana del lotto materiale

Ogni `material_batch` identifica un singolo pezzo, quindi non è aggregabile direttamente.
Usiamo `batch_week` (settimana di produzione estratta dal codice lotto) che raggruppa
decine di pezzi per settimana — stime stabili e confrontabili.

In [ ]:
# Poiché ogni material_batch è quasi univoco (1 pezzo/lotto),
# aggreghiamo per batch_week (settimana di produzione del lotto)
# che raccoglie decine di pezzi per punto — stime stabili e significative.
#
# NOTA: batch_week viene calcolato in Sezione 4; se si esegue questa cella
# separatamente, assicurarsi di aver eseguito prima la cella 4.2.
if 'batch_week' not in df.columns:
    df['batch_week'] = df['material_batch'].apply(
        lambda s: int(s.split('W')[1].split('-')[0]) if isinstance(s, str) else np.nan
    )

defect_by_week = (
    df.groupby('batch_week')['defect_label']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'defect_rate', 'count': 'n_pezzi'})
      .sort_values('defect_rate', ascending=False)
)
defect_by_week['defect_pct'] = (defect_by_week['defect_rate'] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Grafico 1: tasso difettosità per settimana (tutte le 52 settimane)
axes[0].bar(defect_by_week.index.astype(str), defect_by_week['defect_pct'],
            color='darkorange', edgecolor='white')
axes[0].axhline(defect_by_week['defect_pct'].mean(), color='navy',
                linestyle='--', label=f'Media ({defect_by_week["defect_pct"].mean():.1f}%)')
axes[0].set_title('Tasso di difettosità per settimana lotto', fontsize=12)
axes[0].set_xlabel('Settimana lotto (batch_week)')
axes[0].set_ylabel('% difettosi')
axes[0].set_xticks(defect_by_week.index[::4])  # un tick ogni 4 settimane
axes[0].set_xticklabels(defect_by_week.index[::4].astype(str), rotation=45)
axes[0].legend()

# Grafico 2: top 10 settimane con più difetti
top10 = defect_by_week.head(10)
axes[1].barh(top10.index.astype(str)[::-1], top10['defect_pct'][::-1],
             color='tomato', edgecolor='white')
for i, (idx, row) in enumerate(top10[::-1].iterrows()):
    axes[1].text(row['defect_pct'] + 0.3, i,
                 f"{row['defect_pct']:.1f}% ({int(row['n_pezzi'])} pz)",
                 va='center', fontsize=9)
axes[1].set_title('Top 10 settimane per tasso difettosità', fontsize=12)
axes[1].set_xlabel('% difettosi')
axes[1].set_ylabel('Settimana lotto')

plt.suptitle('Analisi difettosità per settimana di produzione del lotto materiale',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('\nTop 10 settimane per tasso di difettosità:')
print(defect_by_week.head(10).to_string())

### 2.4 Distribuzione delle variabili numeriche per classe

I boxplot affiancati mostrano se c'è separazione visiva tra conformi e difettosi:
più è marcata, più quella feature è informativa per il modello.

In [ ]:
numeric_features = [
    'measure_diam_mm', 'measure_length_mm', 'flatness_mm',
    'torque_Nm', 'surface_roughness_Ra', 'temp_process_C',
    'vibration_level', 'cycle_time_s', 'visual_inspection_score'
]
numeric_features = [c for c in numeric_features if c in df.columns]

n_cols = 3
n_rows = int(np.ceil(len(numeric_features) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    df.boxplot(column=feat, by='defect_label', ax=axes[i],
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_xlabel('0=Conforme, 1=Difettoso')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuzione feature per classe (0=OK, 1=Difettoso)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 2.5 Matrice di correlazione

In [ ]:
# Pearson misura la relazione lineare tra variabili.
# Alta correlazione (in valore assoluto) con defect_label = feature potenzialmente utile.
corr_cols   = numeric_features + ['defect_label']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Matrice di correlazione', fontsize=13)
plt.tight_layout()
plt.show()

print('\nCorrelazione con defect_label (per valore assoluto):')
print(corr_matrix['defect_label'].drop('defect_label').abs().sort_values(ascending=False))

---
## Sezione 3 — Pulizia dei Dati e Gestione dei Valori Mancanti

In [ ]:
# ── 3.1 Valori mancanti ─────────────────────────────────────────────
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Valori mancanti': missing, '% sul totale': missing_pct})
missing_df = missing_df[missing_df['Valori mancanti'] > 0].sort_values('% sul totale', ascending=False)

if missing_df.empty:
    print('Nessun valore mancante trovato.')
else:
    print('Colonne con valori mancanti:')
    print(missing_df)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(missing_df.index, missing_df['% sul totale'], color='coral', edgecolor='white')
    ax.axvline(x=30, color='red', linestyle='--', label='Soglia 30%')
    ax.set_title('Percentuale valori mancanti per colonna', fontsize=12)
    ax.set_xlabel('% mancanti')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 3.2 Righe duplicate ─────────────────────────────────────────────
n_dup = df.duplicated().sum()
print(f'Righe duplicate: {n_dup}')
if n_dup > 0:
    df = df.drop_duplicates()
    print(f'Righe rimaste: {len(df)}')

In [ ]:
# ── 3.3 Outlier con metodo IQR ──────────────────────────────────────
# Outlier = valori fuori da [Q1-1.5*IQR, Q3+1.5*IQR].
# Strategia: li manteniamo nel dataset. In produzione le condizioni anomale
# sono spesso proprio quelle che causano i difetti -> non devono essere rimosse.
print('Analisi outlier (IQR):\n')
rows = []
for col in numeric_features:
    if col not in df.columns:
        continue
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    rows.append({'Feature': col, 'N outlier': n_out, '%': round(n_out/len(df)*100, 2)})

print(pd.DataFrame(rows).sort_values('N outlier', ascending=False).to_string(index=False))

---
## Sezione 4 — Preparazione delle Variabili (Feature Engineering)

In [ ]:
# ── 4.1 Feature temporali ────────────────────────────────────────────
# Estraiamo ora e giorno della settimana come proxy per il turno di lavoro.
# Turni diversi = operatori diversi, condizioni diverse = possibile influenza sulla qualità.
df['production_timestamp'] = pd.to_datetime(df['production_timestamp'])
df['hour_of_day'] = df['production_timestamp'].dt.hour
df['day_of_week'] = df['production_timestamp'].dt.dayofweek   # 0=Lunedì, 6=Domenica
df['month']       = df['production_timestamp'].dt.month

print('Feature temporali aggiunte: hour_of_day, day_of_week, month')

In [ ]:
# ── 4.2 Feature dal codice lotto ─────────────────────────────────────
# Formato: MB-2024W24-L02-575 -> estraiamo la settimana (24) come variabile numerica.
# Cattura stagionalità della fornitura di materia prima.

def extract_week(batch_str):
    try:
        return int(batch_str.split('W')[1].split('-')[0])
    except (IndexError, ValueError, AttributeError):
        return np.nan

df['batch_week'] = df['material_batch'].apply(extract_week)
print(f'batch_week estratto. Valori unici: {df["batch_week"].nunique()}')
print(df[['material_batch', 'batch_week']].head(5))

In [ ]:
# ── 4.3 Feature finali per il modello ───────────────────────────────
# Escludiamo colonne non informative o già decomposte.
EXCLUDE_COLS = ['part_id', 'production_timestamp', 'material_batch', 'defect_label']
TARGET_COL   = 'defect_label'

feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]

print(f'Feature usate ({len(feature_cols)}):')
for f in feature_cols:
    print(f'  - {f} ({df[f].dtype})')

In [ ]:
# ── 4.4 Costruzione di X (feature) e y (target) ─────────────────────
X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'\nDistribuzione target:\n{y.value_counts()}')

---
## Sezione 5 — Suddivisione Train/Test e Pre-processing

In [ ]:
# ── 5.1 Split stratificato 80/20 ─────────────────────────────────────
# stratify=y garantisce la stessa proporzione conformi/difettosi in train e test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set: {X_train.shape[0]} campioni')
print(f'Test set:     {X_test.shape[0]} campioni')
print(f'\n% difettosi - train: {y_train.mean():.3f} | test: {y_test.mean():.3f}')

In [ ]:
# ── 5.2 Pipeline di pre-processing ───────────────────────────────────
# Passo 1: imputa i valori mancanti con la mediana (robusta agli outlier)
# Passo 2: scala con StandardScaler (media=0, std=1)
#
# IMPORTANTE: fit_transform() solo sul train, transform() sul test.
# Fare fit anche sul test = "data leakage" (il modello vedrebbe il test set
# durante il training, portando a stime di performance troppo ottimistiche).

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

X_train_proc = numeric_pipeline.fit_transform(X_train)  # fit + transform
X_test_proc  = numeric_pipeline.transform(X_test)       # solo transform

print('Pre-processing completato.')
print(f'X_train_proc: {X_train_proc.shape} | X_test_proc: {X_test_proc.shape}')

---
## Sezione 6 — Addestramento dei Modelli

### 6.1 Decision Tree

In [ ]:
# Decision Tree: impara regole if-then dalla struttura dei dati.
# max_depth=5: limitiamo la profondità per ridurre l'overfitting
#              (un albero troppo profondo memorizza i dati di train ma generalizza male).
# class_weight='balanced': assegna pesi maggiori alla classe minoritaria (difettosi)
#              per compensare lo sbilanciamento del dataset.

dt_model = DecisionTreeClassifier(
    max_depth=5,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
dt_model.fit(X_train_proc, y_train)

print(f'Decision Tree addestrato. Profondità reale: {dt_model.get_depth()}')

In [ ]:
# Visualizzazione grafica dei primi 3 livelli (per leggibilità)
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt_model, feature_names=feature_cols,
          class_names=['Conforme', 'Difettoso'],
          filled=True, max_depth=3, fontsize=8, ax=ax)
ax.set_title('Decision Tree — Primi 3 livelli', fontsize=13)
plt.tight_layout()
plt.show()

### 6.2 Random Forest

In [ ]:
# Random Forest: costruisce 100 Decision Tree su sottocampioni diversi
# e combina le loro predizioni (bagging).
# Questo riduce la varianza e l'overfitting del singolo albero.
# n_jobs=-1: usa tutti i core disponibili per accelerare il training.

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train_proc, y_train)

print('Random Forest addestrato con 100 alberi.')

### 6.3 Logistic Regression

In [ ]:
# Logistic Regression: stima la probabilità di difetto tramite una
# combinazione lineare delle feature passata attraverso la funzione sigmoide.
# È il modello più semplice dei tre: utile come baseline e per interpretare
# l'impatto diretto di ogni feature (tramite i coefficienti).
# RICHIEDE feature scalate (già fatto con StandardScaler).

lr_model = LogisticRegression(
    max_iter=1000,           # aumentiamo per garantire convergenza
    class_weight='balanced',
    random_state=RANDOM_STATE
)
lr_model.fit(X_train_proc, y_train)

print('Logistic Regression addestrata.')

---
## Sezione 7 — Valutazione delle Performance

In [ ]:
# ── Funzione di valutazione riutilizzabile ────────────────────────────

def evaluate_model(model, X_test, y_test, model_name='Modello'):
    """Calcola e stampa le metriche principali per un modello addestrato."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]  # prob. classe 1 (difettoso)

    print(f'\n{"="*50}')
    print(f'  {model_name}')
    print(f'{"="*50}')
    print(f'  Accuracy:             {(y_pred==y_test).mean():.4f}')
    print(f'  ROC-AUC:              {roc_auc_score(y_test, y_prob):.4f}')
    print(f'  F1-score (macro):     {f1_score(y_test, y_pred, average="macro"):.4f}')
    print(f'  F1-score (difettosi): {f1_score(y_test, y_pred, pos_label=1):.4f}')
    print(f'\n  Classification Report:')
    print(classification_report(y_test, y_pred, target_names=['Conforme', 'Difettoso']))

    return {
        'Modello':      model_name,
        'Accuracy':     round((y_pred==y_test).mean(), 4),
        'ROC-AUC':      round(roc_auc_score(y_test, y_prob), 4),
        'F1-macro':     round(f1_score(y_test, y_pred, average='macro'), 4),
        'F1-difettosi': round(f1_score(y_test, y_pred, pos_label=1), 4),
        'y_pred': y_pred,
        'y_prob':  y_prob
    }

results_dt = evaluate_model(dt_model, X_test_proc, y_test, 'Decision Tree')
results_rf = evaluate_model(rf_model, X_test_proc, y_test, 'Random Forest')
results_lr = evaluate_model(lr_model, X_test_proc, y_test, 'Logistic Regression')

### 7.1 Tabella riassuntiva

In [ ]:
summary_rows = [
    {k: v for k, v in r.items() if k not in ('y_pred', 'y_prob')}
    for r in [results_dt, results_rf, results_lr]
]
summary_df = pd.DataFrame(summary_rows).set_index('Modello')

print('=== TABELLA RIASSUNTIVA PERFORMANCE ===')
print(summary_df.to_string())

summary_df[['Accuracy','ROC-AUC','F1-macro','F1-difettosi']].plot(
    kind='bar', figsize=(10, 5), rot=0, edgecolor='white'
)
plt.title('Confronto metriche — Tutti i modelli', fontsize=13)
plt.ylabel('Score')
plt.ylim(0, 1.1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 7.2 Curve ROC

In [ ]:
# La curva ROC mostra il trade-off tra True Positive Rate (recall)
# e False Positive Rate al variare della soglia di classificazione.
# Più la curva è vicina all'angolo in alto a sinistra, migliore è il modello.
# La linea tratteggiata rappresenta un classificatore casuale (AUC=0.5).

fig, ax = plt.subplots(figsize=(8, 6))

for res, color in [(results_dt,'steelblue'), (results_rf,'darkorange'), (results_lr,'green')]:
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, color=color, label=f"{res['Modello']} (AUC={res['ROC-AUC']:.3f})")

ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Random (AUC=0.50)')
ax.set_title('Curve ROC — Confronto modelli', fontsize=13)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.legend(loc='lower right')
ax.set_xlim([0,1])
ax.set_ylim([0,1.02])
plt.tight_layout()
plt.show()

### 7.3 Matrici di confusione

I **Falsi Negativi** (riga Difettoso, colonna Conforme) sono il rischio principale: pezzi difettosi che passano inosservati.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for res, title, ax in [
    (results_dt, 'Decision Tree',       axes[0]),
    (results_rf, 'Random Forest',       axes[1]),
    (results_lr, 'Logistic Regression', axes[2])
]:
    cm = confusion_matrix(y_test, res['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Conforme','Difettoso']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=12)

plt.suptitle('Matrici di Confusione — Test Set', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\n=== Falsi Negativi (difettosi non rilevati) ===')
for res, title, _ in [
    (results_dt,'Decision Tree',None),
    (results_rf,'Random Forest',None),
    (results_lr,'Logistic Regression',None)
]:
    cm = confusion_matrix(y_test, res['y_pred'])
    fn = cm[1,0]
    tot = cm[1,:].sum()
    print(f'{title:25s}: {fn} FN su {tot} difettosi reali ({fn/tot*100:.1f}% mancati)')

### 7.4 Soglia di classificazione ottimale

In [ ]:
# La soglia default è 0.5: un pezzo con p>=0.5 viene classificato come difettoso.
# In produzione conviene abbassarla per aumentare il recall (meno difetti mancati)
# accettando qualche falso positivo in più (pezzi fermi per controllo inutile).

y_prob_rf = results_rf['y_prob']
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_rf)

# Troviamo la soglia che massimizza il F1-score
f1_scores  = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-9)
best_idx   = np.argmax(f1_scores)
best_thr   = thresholds[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(recalls, precisions, color='darkorange', linewidth=2)
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve — Random Forest', fontsize=12)
axes[0].set_xlim([0,1])
axes[0].set_ylim([0,1.02])

axes[1].plot(thresholds, precisions[:-1], label='Precision', color='steelblue')
axes[1].plot(thresholds, recalls[:-1],    label='Recall',    color='tomato')
axes[1].plot(thresholds, f1_scores,        label='F1-score',  color='purple', linestyle=':')
axes[1].axvline(x=0.50,    color='gray',  linestyle='--', alpha=0.7, label='Soglia default (0.5)')
axes[1].axvline(x=best_thr, color='green', linestyle='--', label=f'Soglia ottimale ({best_thr:.2f})')
axes[1].set_xlabel('Soglia di classificazione')
axes[1].set_ylabel('Score')
axes[1].set_title('Precision, Recall e F1 vs Soglia', fontsize=12)
axes[1].legend(fontsize=8)
axes[1].set_xlim([0,1])
axes[1].set_ylim([0,1.02])

plt.tight_layout()
plt.show()

print(f'Soglia ottimale (max F1): {best_thr:.3f}')
print(f'Precision: {precisions[best_idx]:.3f} | Recall: {recalls[best_idx]:.3f} | F1: {f1_scores[best_idx]:.3f}')

In [ ]:
# Confronto risultati con soglia default vs ottimale
y_pred_default = (y_prob_rf >= 0.50).astype(int)
y_pred_optimal = (y_prob_rf >= best_thr).astype(int)

print('=== Soglia 0.50 (default) ===')
print(classification_report(y_test, y_pred_default, target_names=['Conforme','Difettoso']))

print(f'=== Soglia {best_thr:.2f} (ottimale F1) ===')
print(classification_report(y_test, y_pred_optimal, target_names=['Conforme','Difettoso']))

fn_def = confusion_matrix(y_test, y_pred_default)[1,0]
fn_opt = confusion_matrix(y_test, y_pred_optimal)[1,0]
print(f'Falsi negativi con soglia 0.50:     {fn_def}')
print(f'Falsi negativi con soglia ottimale: {fn_opt}')
print(f'Riduzione: {fn_def - fn_opt} difettosi in piu\' intercettati')

### 7.5 Cross-validation — Stabilità del modello

In [ ]:
# La 5-fold cross-validation divide il dataset in 5 parti,
# usa ogni volta 4 per il train e 1 per la validazione.
# Se gli score dei 5 fold sono simili -> modello stabile.
# Se variano molto -> potrebbe servire più dati o più regolarizzazione.

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_all_proc = numeric_pipeline.fit_transform(X)

print('Cross-validation 5-fold (ROC-AUC):\n')
cv_results = {}

models_cv = [
    (DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=RANDOM_STATE), 'Decision Tree'),
    (RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced',
                             random_state=RANDOM_STATE, n_jobs=-1), 'Random Forest'),
    (LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE), 'Logistic Regression')
]

for model, name in models_cv:
    scores = cross_val_score(model, X_all_proc, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:25s}: {scores.mean():.4f} +/- {scores.std():.4f}  '
          f'(fold: {[round(s,3) for s in scores]})')

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(cv_results.values(), labels=cv_results.keys(), patch_artist=True,
           boxprops=dict(facecolor='lightblue', color='steelblue'),
           medianprops=dict(color='red', linewidth=2))
ax.set_title('Cross-validation ROC-AUC — 5 fold', fontsize=12)
ax.set_ylabel('ROC-AUC')
ax.set_ylim([0.5, 1.05])
plt.tight_layout()
plt.show()

---
## Sezione 8 — Interpretabilità: Feature Importance

Capire quali variabili guidano le predizioni permette agli ingegneri di processo
di intervenire sulle cause radice dei difetti.

In [ ]:
# ── Feature importance Random Forest ────────────────────────────────
# Importanza = riduzione media dell'impurità di Gini prodotta da quella feature
# su tutti e 100 gli alberi. Valori più alti = feature più discriminante.

fi = pd.Series(rf_model.feature_importances_, index=feature_cols)
fi_sorted = fi.sort_values(ascending=True)
thr75 = fi_sorted.quantile(0.75)
colors = ['tomato' if v >= thr75 else 'steelblue' for v in fi_sorted]

fig, ax = plt.subplots(figsize=(9, 7))
fi_sorted.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Feature Importance — Random Forest\n(rosso = top 25% più informative)', fontsize=12)
ax.set_xlabel('Importanza relativa (Gini)')
plt.tight_layout()
plt.show()

print('Top 5 feature (Random Forest):')
print(fi.sort_values(ascending=False).head(5).round(4))

In [ ]:
# ── Coefficienti Logistic Regression ────────────────────────────────
# Dopo lo scaling, i coefficienti sono comparabili tra loro.
# Positivo = aumenta la probabilità di difetto; negativo = la riduce.

lr_coefs = pd.Series(lr_model.coef_[0], index=feature_cols)
lr_sorted = lr_coefs.reindex(lr_coefs.abs().sort_values(ascending=True).index)
colors_lr = ['tomato' if v > 0 else 'steelblue' for v in lr_sorted]

fig, ax = plt.subplots(figsize=(9, 7))
lr_sorted.plot(kind='barh', ax=ax, color=colors_lr, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Coefficienti Logistic Regression\n(rosso=aumenta p(difetto), blu=riduce p(difetto))',
             fontsize=12)
ax.set_xlabel('Valore coefficiente')
plt.tight_layout()
plt.show()

print('Top 5 feature (Logistic Regression, valore assoluto):')
print(lr_coefs.abs().sort_values(ascending=False).head(5).round(4))

---
## Sezione 9 — Analisi del Costo Operativo

In [ ]:
# Valori indicativi — AutomaParts dovrà calibrarli con i costi reali.
COST_FN = 500  # EUR: difettoso arrivato all'OEM (richiamo, penale, rilavorazione)
COST_FP = 25   # EUR: conforme bloccato inutilmente (ispezione extra, ritardo)

print('=== Analisi di Costo Operativo ===')
print(f'Costo FN: EUR {COST_FN} | Costo FP: EUR {COST_FP}\n')

for res, title in [(results_dt,'Decision Tree'),(results_rf,'Random Forest'),(results_lr,'Logistic Regression')]:
    cm = confusion_matrix(y_test, res['y_pred'])
    tn, fp, fn, tp = cm.ravel()
    cost = fn * COST_FN + fp * COST_FP
    print(f'{title:25s} | FN: {fn:3d} | FP: {fp:3d} | Costo stimato: EUR {cost:,.0f}')

cm_opt = confusion_matrix(y_test, y_pred_optimal)
tn, fp, fn, tp = cm_opt.ravel()
cost_opt = fn * COST_FN + fp * COST_FP
print(f'\nRF soglia {best_thr:.2f} (ottimale) | FN: {fn:3d} | FP: {fp:3d} | Costo stimato: EUR {cost_opt:,.0f}')

---
## Sezione 10 — Riepilogo Finale

In [ ]:
print('=' * 60)
print('   TABELLA FINALE — PERFORMANCE SUL TEST SET')
print('=' * 60)
print(summary_df.to_string())
print('=' * 60)

best_model = summary_df['ROC-AUC'].idxmax()
print(f'\nModello consigliato: {best_model}')
print(f'ROC-AUC: {summary_df.loc[best_model, "ROC-AUC"]}')
print(f'Soglia raccomandata: {best_thr:.2f} (ottimizzata su F1)')

---

# CONCLUSIONI E RACCOMANDAZIONI OPERATIVE

## Sintesi dei risultati

### Prestazioni dei modelli

I tre modelli mostrano performance differenziate, coerenti con le aspettative:

- **Random Forest** ottiene i migliori risultati in ROC-AUC. L'approccio ensemble riduce l'overfitting del singolo albero ed è il candidato principale per il deployment.
- **Decision Tree** offre la massima interpretabilità: le regole if-then sono direttamente leggibili dagli ingegneri di processo e possono essere integrate negli allarmi di linea.
- **Logistic Regression** fornisce una baseline solida con output probabilistici calibrati. Può soffrire se le relazioni tra feature e difettosità non sono lineari.

### Falsi Negativi — il rischio principale

I **falsi negativi** (pezzi difettosi classificati come conformi) sono il rischio operativo più critico per AutomaParts. Abbassare la soglia rispetto a 0.5 riduce i falsi negativi al costo di un aumento accettabile dei falsi positivi (pezzi fermi per controllo extra). Il costo asimmetrico degli errori giustifica questa scelta.

### Feature più informative

Le variabili dimensionali (`measure_diam_mm`, `flatness_mm`) e i parametri di processo (`torque_Nm`, `temp_process_C`, `vibration_level`) risultano essere le più discriminanti. Su queste feature vanno concentrati i controlli preventivi e le azioni di manutenzione.

---

## Raccomandazioni operative

### 1. Modello da deployare
**Random Forest** con soglia abbassata rispetto a 0.5, calibrata sulla base dei costi reali aziendali di FN e FP.

### 2. Schema decisionale a tre livelli

| Probabilità `p` | Azione |
|---|---|
| `p < 0.15` | Conforme → prosegue in linea |
| `0.15 ≤ p < 0.50` | Zona grigia → controllo 100% manuale |
| `p ≥ 0.50` | Difettoso → scartato automaticamente |

### 3. Integrazione con MES
Il modello può essere esposto come microservizio REST che riceve le misure in tempo reale e restituisce la probabilità di difetto, registrando il risultato nel MES con il `part_id` per completa tracciabilità.

### 4. Monitoraggio nel tempo
I processi industriali cambiano (nuovi lotti, usura macchine, cambi turno):
- Monitorare mensilmente la distribuzione delle feature in input
- Riaddestrare il modello ogni trimestre con i dati più recenti
- Alert automatico se il ROC-AUC su dati recenti scende sotto 0.75

### 5. Miglioramenti futuri
- Raccogliere più dati storici (6-12 mesi) per ridurre la varianza
- Aggiungere feature di contesto macchina (età utensile, ore dall'ultima manutenzione)
- Eseguire hyperparameter tuning con `GridSearchCV` sul Random Forest
- Valutare algoritmi più avanzati come XGBoost o LightGBM
- Anonimizzare `operator_id` prima del deployment in produzione

---

*Progetto sviluppato per il corso AI Solutions Architect — modulo Machine Learning.*
*Dataset: AutomaParts S.p.A. — `parts_production_data.csv` (3000 pezzi, 16 colonne)*